# Cell Segmentation Pipeline — MERFISH Assignment

**Goal:** Assign ~224K mRNA spots to cells (or background) across 4 test FOVs, measured by Adjusted Rand Index (ARI).

**Approach:**
1. Load raw `.dax` DAPI images; extract middle z-plane (z2) per FOV
2. Fine-tune Cellpose (`nuclei`) on 35 training FOVs using provided GT boundaries as integer masks
3. Run fine-tuned model on each FOV → integer cell mask
4. Assign mRNA spots to cells via pixel lookup (`mask[pixel_y, pixel_x]`)
5. Spots with mask value 0 are labeled background

**Baseline:** Pretrained Cellpose (`nuclei`, diameter=30) ARI ≈ 0.632  
**Target:** Fine-tuning on domain-specific tissue should push ARI > 0.7


## 1. Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, 'src')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score
from cellpose import models as cp_models, train as cp_train

from src.io import load_fov_images
from src.coords import parse_boundary_polygon
from src.train_cellpose import boundaries_to_mask

DATA_ROOT  = "/scratch/pl2820/competition"
PIXEL_SIZE = 0.109
print("Imports OK")


## 2. Data Loading

In [ ]:
# fov_metadata.csv is at the competition root (not in reference/)
meta         = pd.read_csv(f"{DATA_ROOT}/fov_metadata.csv").set_index("fov")
cells        = pd.read_csv(f"{DATA_ROOT}/train/ground_truth/cell_boundaries_train.csv", index_col=0)
spots_train  = pd.read_csv(f"{DATA_ROOT}/train/ground_truth/spots_train.csv")
test_spots   = pd.read_csv(f"{DATA_ROOT}/test_spots.csv")
sub_template = pd.read_csv(f"{DATA_ROOT}/sample_submission.csv")

# Test FOVs are FOV_041–FOV_044
TEST_FOVS = ["FOV_041", "FOV_042", "FOV_043", "FOV_044"]

print(f"Training spots  : {len(spots_train):,}")
print(f"Test spots      : {len(test_spots):,}")
print(f"Template rows   : {len(sub_template):,}")
print(f"Training FOVs   : {spots_train['fov'].nunique()}")
print(f"Test FOVs       : {TEST_FOVS}")


## 3. Exploratory Analysis

In [ ]:
fov_name = "FOV_001"
fov_x = meta.loc[fov_name, "fov_x"]
fov_y = meta.loc[fov_name, "fov_y"]

dapi, polyt = load_fov_images(f"{DATA_ROOT}/train/{fov_name}")
fov_cells = cells[cells.index.str.startswith(fov_name)]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(dapi[2], cmap="gray", vmin=0, vmax=np.percentile(dapi[2], 99))
axes[0].set_title(f"{fov_name} — DAPI z2")

axes[1].imshow(dapi[2], cmap="gray", vmin=0, vmax=np.percentile(dapi[2], 99))
for _cid, row in fov_cells.iterrows():
    xs = str(row.get("boundaryX_z2", "")) if pd.notna(row.get("boundaryX_z2", "")) else ""
    ys = str(row.get("boundaryY_z2", "")) if pd.notna(row.get("boundaryY_z2", "")) else ""
    poly = parse_boundary_polygon(xs, ys)
    if poly is None:
        continue
    xs_px = [(x - fov_x) / PIXEL_SIZE for x in poly.exterior.xy[0]]
    ys_px = [(y - fov_y) / PIXEL_SIZE for y in poly.exterior.xy[1]]
    axes[1].plot(xs_px, ys_px, 'c-', linewidth=0.4, alpha=0.7)
axes[1].set_title(f"{fov_name} — GT boundaries ({len(fov_cells)} cells)")

plt.tight_layout()
plt.savefig("eda_fov001.png", dpi=100)
plt.show()
print(f"FOV_001: {len(fov_cells)} GT cells, {len(spots_train[spots_train['fov']==fov_name]):,} spots")


## 4. Segmentation Model — Cellpose Fine-Tuning

In [ ]:
train_images = []
train_masks  = []
train_fovs   = [f"FOV_{i:03d}" for i in range(1, 36)]

for fov_n in train_fovs:
    fov_dir = f"{DATA_ROOT}/train/{fov_n}"
    if not os.path.exists(fov_dir):
        continue
    try:
        d, _ = load_fov_images(fov_dir)
        fx = meta.loc[fov_n, "fov_x"]
        fy = meta.loc[fov_n, "fov_y"]
        m = boundaries_to_mask(cells, fov_n, fx, fy)
        if m.max() == 0:
            continue
        train_images.append(d[2])
        train_masks.append(m)
    except Exception as exc:
        print(f"  Skipping {fov_n}: {exc}")

print(f"Training on {len(train_images)} FOVs")

os.makedirs("models", exist_ok=True)
# Fine-tune from nuclei model (matches colab baseline model)
base_model = cp_models.CellposeModel(gpu=True, model_type="cyto2")
model_path = cp_train.train_seg(
    base_model.net,
    train_data=train_images,
    train_labels=train_masks,
    channels=[1, 2],
    save_path="models/",
    n_epochs=100,
    learning_rate=0.005,
    weight_decay=1e-5,
    batch_size=8,
    model_name="cellpose_finetuned",
)
print(f"Saved: {model_path}")


## 5. Spot Assignment & Background Labeling

Direct pixel lookup: convert global µm → pixel coords, then read the integer mask value. Mask value 0 = background; 1..N = cell integer IDs.

In [ ]:
finetuned = cp_models.CellposeModel(gpu=True, pretrained_model="models/cellpose_finetuned")

def segment_fov(fov_name, fov_dir):
    """Run fine-tuned Cellpose on DAPI z2. Returns integer mask array."""
    dapi, polyt = load_fov_images(fov_dir)
    pred_masks, _, _ = finetuned.eval(dapi[2], diameter=30, channels=[1, 2])
    return pred_masks

def pixel_lookup_assign(spots_df, masks, fov_x, fov_y):
    """Assign spots to cells via pixel lookup. Returns integer cluster_id array (0=background)."""
    px = np.clip(((spots_df["global_x"].values - fov_x) / PIXEL_SIZE).astype(int), 0, 2047)
    py = np.clip(((spots_df["global_y"].values - fov_y) / PIXEL_SIZE).astype(int), 0, 2047)
    return masks[py, px]

# Demo on FOV_001 vs GT mask
demo_masks = boundaries_to_mask(cells, "FOV_001", meta.loc["FOV_001","fov_x"], meta.loc["FOV_001","fov_y"])
demo_spots = spots_train[spots_train["fov"] == "FOV_001"].copy()
demo_ids = pixel_lookup_assign(demo_spots, demo_masks, meta.loc["FOV_001","fov_x"], meta.loc["FOV_001","fov_y"])
bg_frac = (demo_ids == 0).mean()
print(f"FOV_001 GT mask demo: {demo_masks.max()} cells, {bg_frac:.1%} background")


## 6. Local Validation (held-out FOVs)

In [ ]:
val_fovs = [f"FOV_{i:03d}" for i in range(36, 41)]
ari_scores = {}

for fov_n in val_fovs:
    fov_dir = f"{DATA_ROOT}/train/{fov_n}"
    if not os.path.exists(fov_dir):
        continue

    fx = meta.loc[fov_n, "fov_x"]
    fy = meta.loc[fov_n, "fov_y"]
    pred_masks = segment_fov(fov_n, fov_dir)
    gt_mask    = boundaries_to_mask(cells, fov_n, fx, fy)

    fov_spots = spots_train[spots_train["fov"] == fov_n].copy()
    pred_ids = pixel_lookup_assign(fov_spots, pred_masks, fx, fy)
    gt_ids   = pixel_lookup_assign(fov_spots, gt_mask,   fx, fy)

    ari = adjusted_rand_score(gt_ids, pred_ids)
    ari_scores[fov_n] = ari
    print(f"{fov_n}: ARI = {ari:.4f}  ({pred_masks.max()} cells)")

mean_val_ari = np.mean(list(ari_scores.values())) if ari_scores else 0.0
print(f"\nMean validation ARI : {mean_val_ari:.4f}")
print(f"Baseline (pretrained): 0.632")
print(f"Improvement          : {mean_val_ari - 0.632:+.4f}")


## 7. Test FOV Inference

In [ ]:
all_cluster_ids = {}  # spot_id -> integer cluster_id (0=background)

for fov_n in TEST_FOVS:
    fov_dir = f"{DATA_ROOT}/test/{fov_n}"
    if not os.path.exists(fov_dir):
        print(f"  Missing: {fov_dir}")
        continue
    fx = meta.loc[fov_n, "fov_x"]
    fy = meta.loc[fov_n, "fov_y"]
    pred_masks = segment_fov(fov_n, fov_dir)

    fov_spots = test_spots[test_spots["fov"] == fov_n].copy()
    cluster_ids = pixel_lookup_assign(fov_spots, pred_masks, fx, fy)

    for spot_id, cid in zip(fov_spots["spot_id"].values, cluster_ids):
        all_cluster_ids[spot_id] = int(cid) if cid > 0 else "background"

    bg = (cluster_ids == 0).mean()
    print(f"{fov_n}: {pred_masks.max()} cells, {len(fov_spots):,} spots, {bg:.1%} background")

print(f"\nTotal assigned: {len(all_cluster_ids):,}")


## 8. Submission Generation

In [ ]:
sub = sub_template.copy()
sub["cluster_id"] = sub["spot_id"].map(all_cluster_ids)

null_count = sub["cluster_id"].isna().sum()
if null_count > 0:
    print(f"WARNING: {null_count} spots not in assignments — filling with 0 (background)")
    sub["cluster_id"] = sub["cluster_id"].fillna(0)

sub["cluster_id"] = sub["cluster_id"].astype(int)

assert len(sub) == len(sub_template)
assert list(sub.columns) == ["spot_id", "fov", "cluster_id"]
assert sub["cluster_id"].notna().all()

sub.to_csv("submission.csv", index=False)
print(f"submission.csv saved — {len(sub):,} rows")
print(sub["cluster_id"].value_counts().head(10))
print(sub.head())


## 9. Methods Summary

**Task:** Assign ~224K mRNA spots (4 test FOVs) to segmented cells or background, evaluated by mean ARI.

**Pipeline:**
1. **Data loading** — Raw `.dax` files read as uint16; DAPI channel extracted at z-plane 2 (middle of 5 z-planes, frame index 16)
2. **GT mask generation** — GT boundary polygons (µm) rasterized to integer masks via `skimage.draw.polygon` for Cellpose training supervision
3. **Fine-tuning** — Cellpose `nuclei` fine-tuned for 100 epochs on 35 training FOVs (lr=0.005, batch=8, diameter=30)
4. **Inference** — Fine-tuned model applied to each FOV's DAPI z2; outputs integer mask (0=background, 1..N=cells)
5. **Spot assignment** — Direct pixel lookup: `cluster_id = mask[pixel_y, pixel_x]` where pixel coords come from global µm via `(global_x - fov_x) / 0.109`
6. **Submission** — `submission.csv` with integer `cluster_id` column (0 = background)

**Key parameters:**
- z-plane: 2 (middle), pixel size: 0.109 µm/px, diameter: 30
- Cellpose base model: `nuclei`, epochs: 100
- Train FOVs: 001–035, validation FOVs: 036–040, test FOVs: 041–044

**Baseline ARI (pretrained Cellpose nuclei, diameter=30):** ~0.632  
**Expected improvement:** Domain-specific fine-tuning on 35 FOVs of similar tissue type
